In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

kenpom = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, 
    "nishaanamin/march-madness-data", 
    "KenPom Barttorvik.csv",
    pandas_kwargs={"usecols": ["YEAR", "TEAM", "KADJ EM", "BARTHAG", "SEED", "ROUND", "QUAD ID"]}
)
heat = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "nishaanamin/march-madness-data",
    "Heat Check Ratings.csv",
    pandas_kwargs={"usecols": ["YEAR", "TEAM", "EASY DRAW", "TOUGH DRAW", "DARK HORSE", "UPSET ALERT", "CINDERELLA"]}
)
shoot = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, 
    "nishaanamin/march-madness-data", 
    "Shooting Splits.csv",
    pandas_kwargs={"usecols": ["YEAR", "TEAM", "THREES FG%", "THREES FG%D", "THREES SHARE", "CLOSE TWOS FG%", "CLOSE TWOS FG%D", "DUNKS FG%"]}
)

print("\nKenPom Barttorvik\n", kenpom.head())
print("\nHeat Check Ratings\n", heat.head())
print("\nShooting Splits\n", shoot.head())


KenPom Barttorvik
    YEAR  QUAD ID      TEAM  SEED  ROUND  KADJ EM  BARTHAG
0  2026        1     Akron    12      0  12.7986    0.773
1  2026        1   Alabama     4      0  25.7196    0.934
2  2026        2   Arizona     1      0  37.6556    0.978
3  2026        2  Arkansas     4      0  26.0527    0.934
4  2026        2       BYU     6      0  23.2459    0.887

Heat Check Ratings
    YEAR         TEAM  EASY DRAW  TOUGH DRAW  DARK HORSE  UPSET ALERT  \
0  2026   Louisville      False       False        True        False   
1  2026    Tennessee      False       False        True        False   
2  2025  Connecticut      False       False        True        False   
3  2025     Oklahoma      False        True       False        False   
4  2025      Memphis      False        True       False         True   

   CINDERELLA  
0       False  
1       False  
2       False  
3       False  
4       False  

Shooting Splits
    YEAR      TEAM  DUNKS FG%  CLOSE TWOS FG%  CLOSE TWOS FG%D  T

In [11]:
df = kenpom.merge(heat, on=["YEAR", "TEAM"], how="inner").merge(shoot, on=["YEAR", "TEAM"], how="inner")
df.head()

,YEAR,QUAD ID,TEAM,SEED,ROUND,KADJ EM,BARTHAG,EASY DRAW,TOUGH DRAW,DARK HORSE,UPSET ALERT,CINDERELLA,DUNKS FG%,CLOSE TWOS FG%,CLOSE TWOS FG%D,THREES FG%,THREES SHARE,THREES FG%D
0,2026,4,Louisville,6,0,25.4242,0.936,False,False,True,False,False,88.1,64.7,60.3,35.7,52.8,32.7
1,2026,1,Tennessee,6,0,26.0249,0.941,False,False,True,False,False,89.3,60.8,56.9,33.4,31.7,30.6
2,2025,4,Alabama St.,16,64,-9.2350,0.270,False,True,False,False,False,77.0,53.1,57.7,32.9,42.9,33.4
3,2025,3,Arkansas,10,16,17.6925,0.862,False,True,False,False,False,94.5,64.6,55.7,33.3,36.7,31.9
4,2025,2,Baylor,9,32,21.1589,0.903,False,False,True,False,False,95.6,62.8,68.6,34.7,40.2,35.4


In [14]:
#take out 2026 rows since for these rows, ROUND = 0 (ongoing tournament)
df_past = df[df["ROUND"] != 0].copy()
df_2026 = df[df["ROUND"] == 0].copy()

# current dataset has values in ROUND like 64, 16, 2, etc.
# representing how far the team made it in the tourney (i.e. Round of 64)
# change so value in ROUND_ENCODED shows how many games the team played in the tourney before losing
# e.g. ROUND = 16 -> Round of 16 -> played 3 games (R64, R32, R16) -> ROUND_ENCODED = 3
rounds = {64: 1, 32: 2, 16: 3, 8: 4, 4: 5, 2: 6, 1: 7}
df_past["ROUND_ENCODED"] = df_past["ROUND"].map(rounds)

X = df_past.drop(columns=["ROUND", "ROUND_ENCODED", "TEAM", "YEAR", "QUAD ID"])
y = df_past["ROUND_ENCODED"]
X

,SEED,KADJ EM,BARTHAG,EASY DRAW,TOUGH DRAW,DARK HORSE,UPSET ALERT,CINDERELLA,DUNKS FG%,CLOSE TWOS FG%,CLOSE TWOS FG%D,THREES FG%,THREES SHARE,THREES FG%D
2,16,-9.2350,0.270,False,True,False,False,False,77.0,53.1,57.7,32.9,42.9,33.4
3,10,17.6925,0.862,False,True,False,False,False,94.5,64.6,55.7,33.3,36.7,31.9
4,9,21.1589,0.903,False,False,True,False,False,95.6,62.8,68.6,34.7,40.2,35.4
5,12,17.2994,0.876,True,False,False,False,True,87.5,66.6,60.8,36.6,42.8,33.1
6,8,19.3318,0.883,False,False,True,False,False,95.0,63.8,51.4,35.4,42.6,35.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321,5,17.2254,0.877,False,True,False,False,False,90.1,64.3,55.0,32.6,34.7,31.6
322,5,19.3295,0.917,False,True,False,True,False,88.0,59.8,61.1,35.1,35.7,32.9
323,9,13.6158,0.835,False,True,False,False,False,74.6,55.5,51.8,33.2,35.3,37.2
324,9,17.3895,0.867,False,True,False,False,False,86.8,63.6,53.8,33.9,35.3,32.2


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ct = ColumnTransformer([("scaling", StandardScaler(), ["SEED", "KADJ EM", "BARTHAG", "DUNKS FG%", "CLOSE TWOS FG%", "CLOSE TWOS FG%D", "THREES FG%", "THREES SHARE", "THREES FG%D"]))

# maybe need editing for seed cuz lower seed = better team?